## Perceptron 

In [2]:
import torch
class Perceptron:
    def __init__(self, input_dim, learning_rate=0.1):
        self.learning_rate = learning_rate
        self.weights = torch.randn(input_dim + 1)

    def _augment(self, x):
        return torch.cat([torch.tensor([1.0]), x])

    def step(self, z):
        return 1.0 if z >= 0 else 0.0

    def predict(self, x):
        x = self._augment(x)
        z = torch.dot(self.weights, x)
        return self.step(z)

    def fit(self, X, y, epochs=10):

        for epoch in range(epochs):
            errors = 0
            for x, target in zip(X, y):

                x_aug = self._augment(x)
                prediction = self.step(torch.dot(self.weights, x_aug))
                error = target - prediction
                if error != 0:
                    self.weights += self.learning_rate * error * x_aug
                    errors += 1
            print(f"Epoch {epoch+1:2d} | Errors: {errors} | Weights: {self.weights}")

            if errors == 0:
                print("Training converged.")
                break

In [3]:
import torch

X = torch.tensor([
    [1., 45.],
    [2., 50.],
    [2., 55.],
    [3., 60.],
    [4., 65.],
    [5., 70.],
    [6., 75.],
    [7., 80.],
    [8., 85.],
    [9., 90.]
])

y = torch.tensor([
    0., 0., 0., 0., 0.,
    1., 1., 1., 1., 1.
])

In [4]:
model = Perceptron(
    input_dim=2,
    learning_rate=0.01
)

model.fit(X, y, epochs=100)

Epoch  1 | Errors: 1 | Weights: tensor([-0.1861,  0.6212,  0.3332])
Epoch  2 | Errors: 2 | Weights: tensor([-0.1861,  0.6612,  0.5832])
Epoch  3 | Errors: 3 | Weights: tensor([-0.1961,  0.6812,  0.3332])
Epoch  4 | Errors: 2 | Weights: tensor([-0.1961,  0.7212,  0.5832])
Epoch  5 | Errors: 3 | Weights: tensor([-0.2061,  0.7412,  0.3332])
Epoch  6 | Errors: 2 | Weights: tensor([-0.2061,  0.7812,  0.5832])
Epoch  7 | Errors: 3 | Weights: tensor([-0.2161,  0.8012,  0.3332])
Epoch  8 | Errors: 2 | Weights: tensor([-0.2161,  0.8412,  0.5832])
Epoch  9 | Errors: 3 | Weights: tensor([-0.2261,  0.8612,  0.3332])
Epoch 10 | Errors: 2 | Weights: tensor([-0.2261,  0.9012,  0.5832])
Epoch 11 | Errors: 3 | Weights: tensor([-0.2361,  0.9212,  0.3332])
Epoch 12 | Errors: 2 | Weights: tensor([-0.2361,  0.9612,  0.5832])
Epoch 13 | Errors: 3 | Weights: tensor([-0.2461,  0.9812,  0.3332])
Epoch 14 | Errors: 2 | Weights: tensor([-0.2461,  1.0212,  0.5832])
Epoch 15 | Errors: 3 | Weights: tensor([-0.2561,

In [5]:
test_samples = torch.tensor([
    [2., 48.],   # Expected Fail
    [3., 72.],   # Borderline
    [6., 78.],   # Expected Pass
    [8., 88.]    # Expected Pass
])

for sample in test_samples:
    print(sample.tolist(), "->", model.predict(sample))

[2.0, 48.0] -> 0.0
[3.0, 72.0] -> 0.0
[6.0, 78.0] -> 1.0
[8.0, 88.0] -> 1.0


## Single Neuron 

In [9]:
import torch

X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0],
    [6.0],
    [7.0],
    [8.0]
])

y = torch.tensor([
    [0.0],
    [0.0],
    [0.0],
    [0.0],
    [1.0],
    [1.0],
    [1.0],
    [1.0]
])

In [10]:
import torch.nn as nn

class SingleNeuron(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(1, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        z = self.linear(x)

        y_hat = self.sigmoid(z)

        return y_hat

In [11]:
model = SingleNeuron()
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)

epochs = 100

for epoch in range(epochs):

    # Forward pass
    predictions = model(X)

    # Compute loss
    loss = criterion(predictions, y)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d} "
            f"Loss = {loss.item():.4f}"
        )      

Epoch  10 Loss = 0.7194
Epoch  20 Loss = 0.6802
Epoch  30 Loss = 0.6444
Epoch  40 Loss = 0.6117
Epoch  50 Loss = 0.5819
Epoch  60 Loss = 0.5547
Epoch  70 Loss = 0.5299
Epoch  80 Loss = 0.5073
Epoch  90 Loss = 0.4866
Epoch 100 Loss = 0.4677


In [12]:
print("Weight:", model.linear.weight)
print("Bias:", model.linear.bias)

Weight: Parameter containing:
tensor([[0.3416]], requires_grad=True)
Bias: Parameter containing:
tensor([-0.9207], requires_grad=True)


In [13]:
test = torch.tensor([
    [2.5],
    [4.5],
    [6.5]
])

with torch.no_grad():

    probabilities = model(test)

    predictions = (probabilities >= 0.5).float()

for x, p, y_hat in zip(test, probabilities, predictions):

    print(
        f"Hours: {x.item():.1f} | "
        f"Probability: {p.item():.3f} | "
        f"Prediction: {int(y_hat.item())}"
    )

Hours: 2.5 | Probability: 0.483 | Prediction: 0
Hours: 4.5 | Probability: 0.649 | Prediction: 1
Hours: 6.5 | Probability: 0.786 | Prediction: 1


## Multi - Layer - Perceptron 

In [14]:
import torch
import torch.nn as nn


class StudentMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(3, 4),
            nn.ReLU(),

            nn.Linear(4, 3),
            nn.ReLU(),

            nn.Linear(3, 1),
            nn.Sigmoid()

        )

    def forward(self, x):

        return self.network(x)

In [15]:
X = torch.tensor([

    [2.,40.,1.],
    [3.,50.,2.],
    [3.,60.,3.],
    [4.,65.,4.],
    [5.,70.,5.],
    [6.,75.,6.],
    [7.,80.,7.],
    [8.,90.,8.]

])

y = torch.tensor([

    [0.],
    [0.],
    [0.],
    [0.],
    [1.],
    [1.],
    [1.],
    [1.]

])

In [16]:
model = StudentMLP()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [17]:
epochs = 500

for epoch in range(epochs):

    predictions = model(X)

    loss = criterion(predictions, y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch+1)%50==0:

        print(
            epoch+1,
            loss.item()
        )

50 0.46020716428756714
100 0.19866807758808136
150 0.11267899721860886
200 0.07100749015808105
250 0.05099349468946457
300 0.04012157768011093
350 0.030536357313394547
400 0.024780914187431335
450 0.020643362775444984
500 0.017295731231570244


In [18]:
test = torch.tensor([

    [2.,45.,2.],
    [4.,68.,5.],
    [6.,78.,6.],
    [8.,92.,8.]

])

with torch.no_grad():

    probability = model(test)

    prediction = (probability >=0.5).float()

for x,pred,prob in zip(test,prediction,probability):

    print(x.tolist(), pred.item(), prob.item())

[2.0, 45.0, 2.0] 0.0 0.032368578016757965
[4.0, 68.0, 5.0] 1.0 0.5618336796760559
[6.0, 78.0, 6.0] 1.0 0.9999661445617676
[8.0, 92.0, 8.0] 1.0 0.9999995231628418
